# PhysioLive

A real-time physiotherapy coach that runs on a laptop CPU. The webcam sees the exercise, the app draws the full-body skeleton with finger joints, counts each rep, and speaks per-rep feedback whenever the form deviates from the target. Sessions are stored on disk and reviewable in the dashboard's Session and Progress tabs.

## How to run

1. `pip install -r requirements.txt` (Python 3.10+).
2. In this notebook, click **Kernel > Restart & Run All**.
3. A browser tab opens at `http://localhost:8000` and shows the live camera with the skeleton overlay, the rep counter, and the coach's feedback.
4. To stop, use **Kernel > Interrupt**.

The first run downloads the pose model on demand (a one-time step).

## Step 1 - Load libraries and locate the project

Adds `src/` to Python's import path and pulls in the app modules that do the heavy lifting: the pose backend, the angle math, the rep counter, the rule engine, the voice worker, the dashboard server, the session log, the retrieval service, and the coach agent. OpenCV is used for camera capture and drawing. Modules that depend on optional dependencies (retrieval, coach) are wrapped in try/except so a partial install still runs the core pipeline.

In [ ]:
import json
import sys
import time
import webbrowser
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = Path().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from app.pose_gate import PoseInferencer, draw_pose, KP_MIN_CONF
from app.angles import all_angles
from app.rep_counter import RepCounter
from app.form_rules import evaluate as evaluate_rules
from app.voice import VoiceWorker
from app.dashboard_server import DashboardServer, STATE
from app.phone_stream import open_source
from app.session_log import SessionLog
from app.rep_classifier import RepClassifier

try:
    from app.rag import RAGService
except Exception:
    RAGService = None

try:
    from app.agents import CoachAgent, CoachRequest
except Exception:
    CoachAgent, CoachRequest = None, None

from app.agents import progress as progress_agent

print(f"project root: {PROJECT_ROOT}")


## Step 2 - Pick an exercise

Each exercise lives in its own JSON file under `src/app/exercises/`. The file defines the pose backend to use, the rep goal, the state-machine thresholds that decide when a rep starts and ends, and the form rules the coach will check. Change `EXERCISE_ID` below to switch exercises without touching any code.

Available IDs today: `squat`, `lunge`, `glute_bridge`, `leg_raise`, `shoulder_abduction`.

In [ ]:
EXERCISE_ID = "squat"

exercise_path = SRC / "app" / "exercises" / f"{EXERCISE_ID}.json"
with open(exercise_path, "r", encoding="utf-8") as f:
    EXERCISE = json.load(f)

print(f"exercise:    {EXERCISE['name']}")
print(f"backend:     {EXERCISE['pose_backend']}")
print(f"rep goal:    {EXERCISE['rep_goal']}")


## Step 3 - Load and warm the pose model

The default backend is MediaPipe Holistic, which returns 33 body landmarks plus 21 landmarks for each hand (finger articulation) and a virtual neck point. A synthetic warm-up frame is passed through the model so the first real frame from the camera does not pay the initialization cost mid-loop.

In [ ]:
pose = PoseInferencer(backend=EXERCISE["pose_backend"], imgsz=384)
_ = pose.infer(np.zeros((384, 384, 3), dtype=np.uint8))
print("pose model ready.")


## Step 4 - Start the dashboard, voice worker, and session log

The dashboard is a tiny HTTP server on `localhost:8000` that streams the annotated frames as MJPEG and serves the live session state, per-rep history, and long-term progress to the browser. The voice worker runs in its own thread and speaks feedback via the system text-to-speech engine (pyttsx3). The session log persists every rep to a local SQLite database at `data/sessions/physiolive.db`. The browser tab opens automatically.

In [ ]:
server = DashboardServer(port=8000, directory=SRC / "web")
url = server.start()

voice = VoiceWorker()
voice.start()

session_log = SessionLog()
session_id = session_log.open_session(exercise=EXERCISE["name"])
STATE.bind_session_log(session_log)

rep_classifier = RepClassifier()
if rep_classifier.is_available():
    print("rep-quality classifier: loaded")
else:
    print("rep-quality classifier: not present (rules-only mode)")

print(f"dashboard:  {url}")
print(f"session id: {session_id}")
try:
    webbrowser.open(url, new=2)
except Exception:
    pass


## Step 5 - (Optional) Enable the coach agent

When the language-model API key is present, the coach turns each rep verdict into a natural single-sentence message grounded in the retrieval store. When the key is missing or the retrieval store is empty, the pipeline still runs and speaks the plain rule message. To enable, set the `ANTHROPIC_API_KEY` environment variable before starting the notebook.

In [ ]:
rag_service = RAGService() if RAGService is not None else None
rag_ready = rag_service.is_ready() if rag_service else False
print(f"retrieval store: {'ready' if rag_ready else 'empty or unavailable'}")

coach = None
if CoachAgent is not None:
    coach = CoachAgent()
    coach.start()
    if coach.api_key:
        print("coach agent: online")
    else:
        print("coach agent: no API key, will use rule messages only")
else:
    print("coach agent: module not installed")


## Step 6 - Open the video source

By default the built-in webcam (index 0) is used. Change `SOURCE` to:

- `1` for a second webcam.
- An HTTP or RTSP URL for a phone stream, e.g. `"http://192.168.1.42:8080/video"` (works with the IP Webcam / DroidCam apps).
- A path to a local video file for offline testing.

A single frame is read to confirm the source works before the main loop starts.

In [ ]:
SOURCE = 0
cap = open_source(SOURCE, width=1280, height=720, fps=30)
ok, probe = cap.read()
if not ok:
    raise RuntimeError(f"cannot read from source {SOURCE!r}")
print(f"source open: frame shape {probe.shape}")


## Step 7 - Live loop

This is the main pipeline. For every camera frame it:

1. Runs the pose backend and computes joint angles.
2. Picks the primary angle for this exercise (for example the knee for a squat, the shoulder for an abduction) and feeds it into the rep counter's state machine.
3. When a rep closes, evaluates the form rules against the rep's angle statistics and produces a verdict. Optionally asks the coach agent for a natural-language message grounded in the retrieval store.
4. Speaks the verdict through the voice worker. A short debounce keeps the coach from repeating itself when the same message would otherwise fire rep after rep.
5. Persists the rep to the session log and pushes a JSON state to `/api/state` so the dashboard's HUD updates in real time.
6. Draws the full skeleton (body, hands, neck) plus the heads-up display on the frame and publishes the annotated JPEG to the MJPEG stream at `/stream.mjpg`.

Interrupt the kernel to stop cleanly; the finally block closes the session, releases the camera, and stops the worker threads.

In [ ]:
rep_def = EXERCISE["rep_definition"]
PRIMARY_NAME = rep_def["primary_angle"]
SECONDARY_NAME = rep_def.get("secondary_angle")

rep_counter = RepCounter(
    standing_deg=rep_def["standing_deg"],
    bottom_deg=rep_def["bottom_deg"],
    hysteresis_deg=rep_def["hysteresis_deg"],
    confirm_frames=rep_def["confirm_frames"],
)

STATE.set_state({
    "running": True,
    "exercise": EXERCISE["name"],
    "rep_count": 0,
    "rep_goal": EXERCISE["rep_goal"],
    "rep_state": rep_counter.state,
    "knee_angle": None,
    "verdict": {"level": "", "text": ""},
})


def _pick_primary_value(angles):
    a = angles.get(PRIMARY_NAME)
    b = angles.get(SECONDARY_NAME) if SECONDARY_NAME else None
    if a is not None and b is not None:
        return a if rep_counter.decreasing and a <= b or (
            not rep_counter.decreasing and a >= b) else b
    return a if a is not None else b


def _metric_family(name):
    if name.startswith("knee"):
        return "knee"
    if name.startswith("hip"):
        return "hip"
    if name.startswith("elbow"):
        return "elbow"
    if name.startswith("shoulder"):
        return "shoulder"
    return None


PRIMARY_FAMILY = _metric_family(PRIMARY_NAME)


def _build_metrics(angles):
    metrics = {}
    for name in ("knee_left", "knee_right"):
        if angles.get(name) is not None:
            metrics.setdefault("knee", angles[name])
            metrics["knee"] = min(metrics["knee"], angles[name])
    for name in ("hip_left", "hip_right"):
        if angles.get(name) is not None:
            metrics.setdefault("hip", angles[name])
            metrics["hip"] = min(metrics["hip"], angles[name]) \
                if rep_counter.decreasing else max(metrics["hip"], angles[name])
    for name in ("elbow_left", "elbow_right"):
        if angles.get(name) is not None:
            metrics.setdefault("elbow", angles[name])
            metrics["elbow"] = min(metrics["elbow"], angles[name])
    for name in ("shoulder_left", "shoulder_right"):
        if angles.get(name) is not None:
            metrics.setdefault("shoulder", angles[name])
            metrics["shoulder"] = max(metrics["shoulder"], angles[name])
    if angles.get("torso_vertical") is not None:
        metrics["torso_vertical"] = angles["torso_vertical"]
    torso_len = angles.get("torso_length")
    side = "left" if PRIMARY_NAME.endswith("left") else "right"
    kt = angles.get(f"knee_over_toe_{side}")
    if kt is not None and torso_len:
        metrics["knee_over_toe_norm"] = kt / torso_len
    return metrics


def _draw_hud(frame, rep_count, rep_goal, primary_val, state_text,
              verdict_text, verdict_level, primary_label):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (360, 130), (16, 20, 28), -1)
    cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)
    cv2.putText(frame, f"Reps  {rep_count} / {rep_goal}", (24, 46),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (232, 236, 241), 2,
                cv2.LINE_AA)
    val_txt = f"{primary_label}  {int(primary_val)}deg" \
        if primary_val is not None else f"{primary_label}  -"
    cv2.putText(frame, val_txt, (24, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2,
                cv2.LINE_AA)
    cv2.putText(frame, f"State {state_text}", (24, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2,
                cv2.LINE_AA)
    if verdict_text:
        color = {"good": (71, 199, 106), "warn": (36, 165, 245),
                 "bad": (68, 68, 239)}.get(verdict_level, (232, 236, 241))
        cv2.rectangle(frame, (10, h - 60), (min(w - 10, 900), h - 10),
                      (16, 20, 28), -1)
        cv2.putText(frame, verdict_text[:70], (24, h - 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2,
                    cv2.LINE_AA)
    return frame


def _label_from_family(family):
    return {"knee": "Knee", "hip": "Hip", "shoulder": "Shoulder",
            "elbow": "Elbow"}.get(family, "Angle")


primary_label = _label_from_family(PRIMARY_FAMILY)
last_verdict_text = ""
last_verdict_level = ""
last_verdict_sources = []
loop_t0 = time.perf_counter()
flush_t = loop_t0
frames_seen = 0


def _handle_coach(resp):
    global last_verdict_text, last_verdict_sources
    if not resp or not resp.text:
        return
    last_verdict_text = resp.text
    last_verdict_sources = resp.source_urls or []
    voice.say(last_verdict_text)


try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        result = pose.infer(frame)
        kps = result.coco17 if result else None
        angles = all_angles(kps) if kps else {}
        primary_val = _pick_primary_value(angles) if angles else None
        metrics = _build_metrics(angles) if angles else {}

        event = rep_counter.update(primary_val, metrics=metrics)
        if event is not None:
            verdict = evaluate_rules(event.sample, EXERCISE["rules"])
            clf_result = rep_classifier.classify(event.sample) \
                if rep_classifier.is_available() else None
            if clf_result is not None:
                verdict.level = clf_result.label if clf_result.confidence > 0.6 \
                    else verdict.level

            last_verdict_text = verdict.text
            last_verdict_level = verdict.level
            last_verdict_sources = []
            voice.say(last_verdict_text)
            session_log.append_rep(event.index, verdict.level,
                                   verdict.text, event.sample)

            if coach is not None and rag_ready and rag_service is not None:
                chunks = rag_service.search_for_verdict(
                    exercise=EXERCISE["name"],
                    verdict_level=verdict.level,
                    verdict_text=verdict.text,
                    angles={"knee_min_deg": event.sample.get_min("knee")},
                    k=4,
                )
                coach.request(CoachRequest(
                    exercise=EXERCISE["name"],
                    verdict_level=verdict.level,
                    verdict_text=verdict.text,
                    angles=metrics,
                    chunks=chunks,
                ), on_response=_handle_coach)

        if result is not None:
            frame = draw_pose(frame, result, min_conf=KP_MIN_CONF)
        frame = _draw_hud(frame, rep_counter.count, EXERCISE["rep_goal"],
                          primary_val, rep_counter.state,
                          last_verdict_text, last_verdict_level,
                          primary_label)

        STATE.set_state({
            "running": True,
            "exercise": EXERCISE["name"],
            "rep_count": rep_counter.count,
            "rep_goal": EXERCISE["rep_goal"],
            "rep_state": rep_counter.state,
            "knee_angle": primary_val,
            "verdict": {
                "level": last_verdict_level,
                "text": last_verdict_text,
                "source_url": (last_verdict_sources[0]
                               if last_verdict_sources else None),
            },
        })

        now = time.perf_counter()
        if now - flush_t > 1.0:
            session_log.flush()
            flush_t = now

        ok_enc, jpeg = cv2.imencode(".jpg", frame,
                                    [int(cv2.IMWRITE_JPEG_QUALITY), 72])
        if ok_enc:
            STATE.push_frame(jpeg.tobytes())

        frames_seen += 1
        if frames_seen % 60 == 0:
            fps = frames_seen / max(1e-6, now - loop_t0)
            print(f"~{fps:.1f} fps, reps: {rep_counter.count}")
except KeyboardInterrupt:
    print("interrupted by user.")
finally:
    cap.release()
    voice.stop()
    if coach is not None:
        coach.stop()
    session_log.close_session()
    STATE.set_state({**STATE.get_state(), "running": False})
    print(f"final rep count: {rep_counter.count}")


## Step 8 - End-of-session summary and PDF export

After the loop stops, this cell reads the finished session back from the session log, prints the summary numbers, and (if `reportlab` is installed) writes a one-page PDF report under `data/sessions/`. The PDF can be shared with a physiotherapist as an objective record of the session.

In [ ]:
session_summary = session_log.past_sessions(limit=1)["sessions"]
if session_summary:
    print(session_summary[0])

current = session_log.current_session_summary()
if current:
    stats = progress_agent.summarise(current)
    print("session stats:", stats)
    pdf_path = PROJECT_ROOT / "data" / "sessions" / (
        f"session_{current['id'][:8]}.pdf")
    written = progress_agent.export_pdf(current, pdf_path)
    if written:
        print(f"PDF: {written}")
